# Part I

## Python libraries and functions from assignment 0 needed.

In [3]:
#pip install import-ipynb
import import_ipynb
import random as rd
from math import sqrt

In [4]:
# Finds the m- ary expansion of n
def getExpansion (n ,m):
    listOfDigits =[]
    while n >= m:
        digit =n%m
        listOfDigits . append ( digit )
        n =(n - digit ) // m
    listOfDigits.append(n)
    return listOfDigits
    
def intToText (n):
    t= getExpansion(n ,256)
    myString =''
    for i in t :
        myString = myString + chr(i)
    return myString

def textToInt (s):
    n =0
    k =0
    for i in s :
        n=n +ord( i) *(256** k)
        k=k +1
    return n

# Output the multiplicative inverse of a modulo p
def multInverse (a , p):
    result = extendedGCD (a , p)
    if result [0]!=1: # Error message if a and p are not relatively prime
        s=" Numbers needs to be relatively prime "
        return s
    inv = result [1]% p
    return inv

#extended euclidean algorithm
# Output [r,s,t] satisfying s*a+t*b=r=gcd(a,b)
def extendedGCD(a , b):
    r0 , r=a ,b
    s0 , s =1 ,0
    t0 , t =0 ,1
    while (r >0):
        tempr , temps , tempt =r ,s ,t
        q= r0 // r
        r ,s , t=r0 - q*r ,s0 -q*s ,t0 - q*t
        r0 , s0 , t0 = tempr , temps , tempt
    return [r0 ,s0 , t0 ]

def fast2Power (a ,n ,m):
    res =1
    while n >0:
        if n %2==1: #If the bit is 1 multiply by the corresponding square
            res =( res *a )%m
        a =( a*a) %m
        n=n //2
    return res

## Implementation part.

### Exercise 1. The Affine Cipher.

In [7]:
def affineEncrypt(P,a,b,m):
    encrypted = (a * P + b) % m
    return encrypted

def affineDecrypt(C,a,b,m):
    aInverse = multInverse(a,m)
    decryptC = (aInverse * C - aInverse * b) % m
    return decryptC

#### Use case no.1: Test a string of size less than 16.

In [9]:
m = 340282366920938463463374607431768211507 # A prime number.
a = 55451395049073
b = 819472983029
message = "iamintrigued"
messageInt = textToInt(message)
encrypted = affineEncrypt(messageInt, a,b,m)
print("Integer of the message:", messageInt)
print("Encrypted message:", encrypted)

Integer of the message: 31071156914406983298588696937
Encrypted message: 89372971806264394073030374297257912489


In [10]:
decrypted = affineDecrypt(encrypted,a,b,m)
print("Encrypted message:", decrypted)

Encrypted message: 31071156914406983298588696937


In [11]:
decryptedText = intToText(decrypted)
print("Decrypted message:", decryptedText)

Decrypted message: iamintrigued


#### Use case no2: Test a string of length 20.

In [13]:
message = "iamintriguedbycrypto"
messageInt = textToInt(message)
encrypted = affineEncrypt(messageInt,a,b,m)
decrypted = affineDecrypt(encrypted,a,b,m)
originalText = intToText(decrypted)

print("Original message:", message)
print("Integer of the message:", messageInt)
print("Prime:", m)
print("Encrypted message:", encrypted)
print("Decrypted message:", originalText)

Original message: iamintriguedbycrypto
Integer of the message: 636294659906714984930944097310966722482599190889
Prime: 340282366920938463463374607431768211507
Encrypted message: 294081227794375361707406692067460321437
Decrypted message: Nù:5Xtriguedbycr


#### Use case no.3: Decrypting your message using a given "a" and "b".

In [15]:
a = 21712651827515182
b = 274812741241924192499
inputCiphTxt = 306223679614151152019567562383624433868

In [16]:
decrypted = affineDecrypt(inputCiphTxt,a,b,m)
message = intToText(decrypted)
print("Decrypted message:", message)

Decrypted message: This is wrong!


### Exercise 2. Diffie-Hellman.

In [18]:
def diffieHellman(p, g, a):
    """
    Diffie-Hellman's protocol.

    :param p: a prime,
    :param g: a primitive root modulo p,
    :param a: a random number
    :return: raise g to the power of a modulo p.
    """
    return fast2Power(g,a,p)

#### A)

In [20]:
def testDiffieHellman(p, alpha, beta, a, b):
    """
    Test if the Diffie-Hellman's keys match.
    
    :param p: a prime,
    :param alpha: Alice's power,
    :param beta: Bob's power,
    :param a: Alice's random number,
    :param b: Bob's random number,
    :return: string containing the key if they match or a warning otherwise.
    """
    kAlice = fast2Power(beta,a,p)
    kBob = fast2Power(alpha,b,p)
    
    sameKey = kAlice == kBob
    
    if sameKey:
        message = "The shared key is: " + str(kAlice)
    else:
        message = "The keys don't match"
    
    return message


In [21]:
p = 2712691
g = 2
a = 2553
b = 26511

alpha = diffieHellman(p,g,a)
beta = diffieHellman(p,g,b)

result = testDiffieHellman(p,alpha,beta,a,b)
print(result)

The shared key is: 2329035


#### B)

In [23]:
p = 340282366920938463463374607431768211507
g = 121770841829326452690259895862715513623
a = 21712651827515182
b = 274812741241924192499

alpha = diffieHellman(p,g,a)
beta = diffieHellman(p,g,b)

result = testDiffieHellman(p,alpha,beta,a,b)
print(result)

The shared key is: 211724440447734875491018020750812830081


### Exercise 3. Elgamal.

#### A)

In [26]:
def elgamalEncrypt(p,g,A,m):
    """
    Encrypts using Elgamal's protocol

    :param p: a prime number,
    :param g: a primiteive root modulo p,
    :param A: the public key,
    :param m: the message
    :return: List of two integers: Elgamal's ciphertext [c1,c2]
    """
    k = rd.randint(2, p-2) # ephemeral key
    c1 = fast2Power(g,k,p)
    c2 = m*fast2Power(A,k,p) % p
    ciphertext = [c1,c2]
    return ciphertext

def elgamalDecrypt(p,g,a,C):
    """
    Decrypts Elgamal's cyphertext

    :param p: a prime number,
    :param g: a primiteive root modulo p,
    :param a: your private key,
    :param C: Elgamal's list
    :return: an integer with the decrypted message
    """
    c1 = C[0]
    c2 = C[1]
    return c2 * fast2Power(multInverse(c1,p),a,p) % p

#### B)

In [28]:
p = 2189248127867
g = 1267362
a = 28127481748
A = fast2Power(g,a,p)
print ("Elgamal's public key is:", A)

Elgamal's public key is: 1244880003213


In [29]:
text = "hello"
m = textToInt(text)
C = elgamalEncrypt(p,g,A,m)
print("Elgamal's encrypted cyphertext is:", C)


Elgamal's encrypted cyphertext is: [1810735594344, 1163172653051]


#### C)

In [31]:
m = elgamalDecrypt(p,g,a,C)
original = intToText(m)
print("Elgamal's decrypted cyphertext is:", original)

Elgamal's decrypted cyphertext is: hello


#### D)

In [33]:
p = 286197821500073589067075140527
g = 195051999861158821970552089016
a = 7712654356741 # my private key
C = [223905580785475751236633059935, 197494619985708086124722797272]

m = elgamalDecrypt(p,g,a,C)
mText = intToText(m)
print("Elgamal's decrypted cyphertext is:", mText)

Elgamal's decrypted cyphertext is: Very good!


### Exercise 4. Elgamal's digital signature.

#### A)

In [36]:
def generateElgamalSig(p,g):
    """
    Generates private-public key for digital signature.

    :param p: a safe prime (2*q + 1),
    :param g: a primitive root of period q
    :return: a list of integers representing private key-public verification key
    """
    a = rd.randint(2,p-2)
    A = fast2Power(g,a,p)
    return [a,A]
    

#### B)

In [38]:
def signElgamal(a,p,g,D):
    """
    Signs a document
    
    :param a: an integer between 2 and p-2. This is the private key,
    :param p: a safe prime, a prime of the form 2*q + 1 with q prime,
    :param g: a primitive root of period q,
    :param D: an integer represinting a document,
    :return: a list [D,[S1,S2]] where D represents the input document and S1,S2 are integers such that [S1,S2] represents the signature.
    """
    q = int((p-1)/2)       
    k = rd.randint(1, q-1) # note: it's neccessary that k has a multiplicative inverse mod q
    while extendedGCD(k, p-1)[0] != 1:
        k = rd.randint(1, q-1)    
    S1 = fast2Power(g,k,p)
    S2 = ((D - a*S1) * multInverse(k,p - 1)) % (p - 1)
    return [D,[S1,S2]]

def verifyElgamal(A,p,g,D,S):
    """
    Verifies a digital signature

    :param A: integer; the public verification key,
    :param p: integer; a prime,
    :param g: integer; a primitive root modulo p,
    :param D: integer; the document,
    :param S: list of integers of length two; the signature
    :return: Boolean; True if the signature matches, False otherwise.
    """
    return (fast2Power(A,S[0],p) * fast2Power(S[0],S[1],p)) % p == fast2Power(g,D,p)

#### C)

In [40]:
def testElgamalSignature(p,g,m):
    """
    Tests Elgamal's digital signature implementation

    :param p: integer; a safe prime,
    :param g: integer; a primitive root modulo p,
    :param m: a message not too large,
    :return: Boolean; True if the signature matches, False otherwise.    
    """
    D = textToInt(m)
    if D >= p:
        s = "Value D must be smaller than p!"
        return s
    keys = generateElgamalSig(p, g)
    mSigned = signElgamal(keys[0], p, g, D)
    mVerify = verifyElgamal(keys[1], p, g, D, mSigned[1])
    return mVerify

In [41]:
p = 103689462716838947264087359614791642504562258733593955922392638318956512079519
g = 38800622773637695321106177828004110206779343423539510992140900880451899659111

m = "hello world!"

testElgamalSignature(p,g,m)

True

### Exercise 5. Shank's Babystep-Giantstep

#### A)

In [44]:
def babyGiant (g ,h ,p ,N = -1) :
    if N == -1: #If we do not know the order we use N=p -1
        N=p -1
    n= int ( sqrt (N )) +1
    gb =1
    i =0
    babyStepTable ={ gb :i }
    while i <n -1:
        gb =( gb *g)% p
        i=i +1
        babyStepTable [ gb ]= i # Add a new element to the table
    G= fast2Power ( multInverse (g ,p) ,n ,p)
    y=h
    if y in babyStepTable :
        return babyStepTable [y]
    for j in range (1 , n):
        y =( y*G) %p
        if y in babyStepTable :
            return j*n+ babyStepTable [ y]
    return None #If no match is found

In [45]:
p = 3079
h = 2861
g = 26
x = babyGiant(g ,h ,p)
x

101

In [46]:
def testBabyGiant(x,p,g,h):
    return h == fast2Power(g,x,p)

In [47]:
testBabyGiant(x,p,g,h)

True

#### B)

In [49]:
p2 = 270826551115777
g2 = 126736225125251
h2 = 159906865261704
x2 = babyGiant (g2 ,h2 ,p2)
x2

1274182741627

In [50]:
testBabyGiant(x2,p2,g2,h2)

True